In [1]:
import numpy as np
import pandas as pd
import torch 
import cv2
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import os
import h5py
import pims
from tqdm import tqdm
from torchvision.io import read_video
import gc
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torchvision.models as models

In [2]:
VIDEO_BASE_PATH = r"E:\nexar_dataset\train"

meta_pos = pd.read_csv(r"E:\nexar_dataset\metadata1.csv")
meta_neg = pd.read_csv(r"E:\nexar_dataset\metadata2.csv")

meta_pos["target"] = 1
meta_neg["target"] = 0
train = pd.concat([meta_pos, meta_neg], ignore_index=True)
train["id"] = range(len(train))


train["video_path"] = train.apply(
    lambda row: os.path.join(
        VIDEO_BASE_PATH,
        "positive" if row["target"] == 1 else "negative",
        row["file_name"]
    ),
    axis=1)

train["id"] = range(len(train))


train_df, val_df = train_test_split(train, test_size=0.2, stratify=train["target"], random_state=42)


In [3]:
subset_size = int(0.2 * len(train_df))  # 20%
random_subset = train_df.sample(n=subset_size, random_state=42)

print(f"Selected {len(random_subset)} videos for random (no-metadata) frame extraction.")
RANDOM_TRAIN_PATH = r"E:\nexar_dataset\noMeta_train"
os.makedirs(RANDOM_TRAIN_PATH, exist_ok=True)

Selected 240 videos for random (no-metadata) frame extraction.


In [4]:
import pims, cv2, numpy as np, torch, os
from tqdm import tqdm

def extract_random_frames(df, num_frames=32, output_path=RANDOM_TRAIN_PATH):
    os.makedirs(output_path, exist_ok=True)

    for i in tqdm(range(len(df)), desc="Extracting Random Frames", ncols=100):
        sample = df.iloc[i]
        path = sample["video_path"]

        try:
            video = pims.Video(path)
        except Exception as e:
            print(f"Error in video {path}: {e}")
            continue

        total_frames = len(video)
        if total_frames < 1:
            continue

        indices = np.linspace(0, total_frames - 1, num_frames).astype(int)

        frames = []
        for idx in indices:
            frame = np.array(video[idx])
            frame = cv2.resize(frame, (224, 224))
            if frame.ndim == 2:
                frame = np.stack([frame]*3, axis=-1)
            frame = frame.astype("float32") / 255.0
            frame = torch.tensor(frame).permute(2, 0, 1)
            frames.append(frame)

        if len(frames) == 0:
            continue

        frames_np = torch.stack(frames).numpy()
        np.save(os.path.join(output_path, f"{int(sample['id'])}"), frames_np)


In [10]:
extract_random_frames(random_subset, num_frames=32, output_path=RANDOM_TRAIN_PATH)


Extracting Random Frames:   0%|                                             | 0/240 [00:00<?, ?it/s]

Extracting Random Frames: 100%|█████████████████████████████████| 240/240 [6:23:39<00:00, 95.92s/it]


In [6]:
from torch.utils.data import Dataset
from torchvision import transforms

class CrashFrameDataset(Dataset):
    def __init__(self, df, frame_dirs, num_frames=32):
        self.df = df.reset_index(drop=True)
        if isinstance(frame_dirs, str):
            self.frames_dirs = [frame_dirs]
        else:
            self.frames_dirs = frame_dirs
        self.num_frames = num_frames

        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
        
    def __getitem__(self, idx):
        sample = self.df.iloc[idx]
        video_id = int(sample["id"])
        target = int(sample["target"])

        npy_path = None
        for d in self.frames_dirs:
            path_candidate = os.path.join(d, f"{video_id}.npy")
            if os.path.exists(path_candidate):
                npy_path = path_candidate
                break

        if npy_path is None:
            print(f" Skipped missing file: {video_id}")
            dummy = torch.zeros((self.num_frames, 3, 224, 224))
            return dummy, torch.tensor(0)

        frames = np.load(npy_path)  # (T, 3, 224, 224)
        T = frames.shape[0]

        if T > self.num_frames:
            idxs = np.linspace(0, T - 1, self.num_frames).astype(int)
            frames = frames[idxs]
        elif T < self.num_frames:
            last = frames[-1][None, ...]
            pad = np.repeat(last, self.num_frames - T, axis=0)
            frames = np.concatenate([frames, pad], axis=0)

        frames = torch.tensor(frames, dtype=torch.float32)
        if frames.ndim == 4 and frames.shape[-1] == 3:
            frames = frames.permute(0, 3, 1, 2)

        frames = frames / 255.0
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                        std=[0.229, 0.224, 0.225])
        frames = normalize(frames)

        label = torch.tensor(target, dtype=torch.long)
        return frames, label

    def __len__(self):
        return len(self.df)


In [7]:
TRAIN_PATH = r"E:\nexar_dataset\new_frames\train"
RANDOM_PATH = r"E:\nexar_dataset\noMeta_train"
VAL_PATH = r"E:\nexar_dataset\new_frames\val"

train_dataset = CrashFrameDataset(train_df, [TRAIN_PATH, RANDOM_PATH], num_frames=32)
val_dataset   = CrashFrameDataset(val_df, [VAL_PATH], num_frames=32)




In [8]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=0)

print(f"Dataset ready! Train videos: {len(train_dataset)}, Validation videos: {len(val_dataset)}")

Dataset ready! Train videos: 1200, Validation videos: 300


In [9]:
import torch
import torch.nn as nn
import torchvision.models as models

class CrashDetectionModel(nn.Module):
    def __init__(self, num_classes=2, hidden_size=256, dropout=0.3):
        super(CrashDetectionModel, self).__init__()

        self.feature_extractor = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.feature_extractor.classifier = nn.Identity()

        for param in self.feature_extractor.parameters():
            param.requires_grad = False

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.pre_lstm = nn.Sequential(
            nn.Linear(1280, 512),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # LSTM
        self.lstm = nn.LSTM(
            input_size=512,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        # Classifier 
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # x: (B, T, 3, 224, 224)
        B, T, C, H, W = x.shape
        x = x.contiguous().view(B * T, C, H, W)  # (B*T, 3, 224, 224)

        feats = self.feature_extractor.features(x)   # (B*T, 1280, h, w)
        feats = self.global_pool(feats).squeeze(-1).squeeze(-1)  # (B*T, 1280)

        feats = feats.view(B, T, 1280)
        feats = self.pre_lstm(feats)  # (B, T, 512)

        lstm_out, _ = self.lstm(feats)  # (B, T, hidden)
        last_output = lstm_out[:, -1, :]  # (B, hidden)

        out = self.classifier(last_output)
        return out


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = CrashDetectionModel(num_classes=2, hidden_size=256, dropout=0.3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)  
num_epochs = 30
best_val_loss = float('inf')
patience = 7
counter = 0


Using device: cuda


In [11]:
from torch.nn.utils import clip_grad_norm_

history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": []
}

best_val_loss = float('inf')
patience = 5
counter = 0

for epoch in range(num_epochs):
    print(f"\n Epoch {epoch+1}/{num_epochs}")
    model.train()
    running_loss = 0.0
    correct_train, total_train = 0, 0

    for frames, labels in tqdm(train_loader, desc=f"Training Epoch {epoch+1}", ncols=100):
        frames = frames.to(device)
        labels = labels.to(device).long()

        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss.backward()

        clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        running_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)
        torch.cuda.empty_cache()

    avg_train_loss = running_loss / len(train_loader)
    train_acc = correct_train / total_train

    model.eval()
    val_loss = 0.0
    correct_val, total_val = 0, 0

    with torch.no_grad():
        for frames, labels in tqdm(val_loader, desc=f"Validating Epoch {epoch+1}", ncols=100):
            frames = frames.to(device)
            labels = labels.to(device).long()

            outputs = model(frames)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)
            torch.cuda.empty_cache()

    avg_val_loss = val_loss / len(val_loader)
    val_acc = correct_val / total_val

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f" Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0
        torch.save(model.state_dict(), "best_model_combined.pth")
        print(" Validation loss improved — model saved.")
    else:
        counter += 1
        print(f" No improvement for {counter} epochs.")
        if counter >= patience:
            print(" Early stopping triggered.")
            break

print("\n Training finished!")



 Epoch 1/30


Validating Epoch 1: 100%|█████████████████████████████████████████| 150/150 [00:27<00:00,  5.48it/s]


 Epoch 1/30 | Train Loss: 0.6917 | Train Acc: 0.5325 | Val Loss: 0.6966 | Val Acc: 0.5100
 Validation loss improved — model saved.

 Epoch 2/30


Validating Epoch 2: 100%|█████████████████████████████████████████| 150/150 [00:29<00:00,  5.10it/s]


 Epoch 2/30 | Train Loss: 0.6699 | Train Acc: 0.5842 | Val Loss: 1.1887 | Val Acc: 0.5000
 No improvement for 1 epochs.

 Epoch 3/30


...

 Epoch 4/30 | Train Loss: 0.6545 | Train Acc: 0.6708 | Val Loss: 1.0329 | Val Acc: 0.5067
 No improvement for 3 epochs.

 Epoch 5/30


Validating Epoch 5: 100%|█████████████████████████████████████████| 150/150 [00:26<00:00,  5.73it/s]


 Epoch 5/30 | Train Loss: 0.6379 | Train Acc: 0.6917 | Val Loss: 1.4926 | Val Acc: 0.5000
 No improvement for 4 epochs.

 Epoch 6/30


Validating Epoch 6: 100%|█████████████████████████████████████████| 150/150 [00:27<00:00,  5.51it/s]

 Epoch 6/30 | Train Loss: 0.6233 | Train Acc: 0.6842 | Val Loss: 1.0799 | Val Acc: 0.4833
 No improvement for 5 epochs.
 Early stopping triggered.

 Training finished!
